# NutriMatch — User Profile Feature Engineering
**Role:** Data Scientist (Orang A — Nutrisi & Profil Pengguna)

**Tujuan notebook ini:**
Menghasilkan `user_profile_features_schema.csv` — skema fitur pengguna yang berisi formula BMR/TDEE, target kalori per goal, target makronutrien harian, dan allergy_vector placeholder.

**Input:** `weight_change_dataset.csv` — 100 baris data profil fisik simulasi (digunakan sebagai sanity check formula, bukan training utama)

**Output:** `user_profile_features_schema.csv`

---
**Catatan penting — Synthetic Data:**
Dataset `weight_change_dataset.csv` tidak memiliki kolom `height_cm`. Nilai tinggi badan **di-generate secara sintetis** menggunakan distribusi normal berdasarkan gender:
- Laki-laki: mean 170 cm, std 7 cm (referensi: rata-rata tinggi pria Indonesia)
- Perempuan: mean 158 cm, std 6 cm (referensi: rata-rata tinggi wanita Indonesia)

Data ini hanya untuk **validasi formula dan demo schema** — bukan untuk training model produksi.

---
**Pipeline:**
1. Import & Load
2. Initial Audit
3. Rename & Type Standardization
4. Unit Conversion (lbs → kg)
5. Synthetic Height Generation
6. BMR Calculation (Mifflin-St Jeor)
7. TDEE Calculation
8. Goal Assignment & Target Calorie
9. Macronutrient Target Calculation
10. Macro Validation
11. Allergy Vector Placeholder
12. Final Export

## 1. Import Library

In [22]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Load Dataset & Initial Audit

In [23]:
df = pd.read_csv('weight_change_dataset.csv')

print('=' * 55)
print('INITIAL AUDIT — weight_change_dataset.csv')
print('=' * 55)
print(f'Shape       : {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Duplicates  : {df.duplicated().sum()}')
print()
print('--- Columns ---')
print(df.columns.tolist())
print()
print('--- Missing Values ---')
print(df.isnull().sum())
print()
print('--------- Sample -----------')
df.head()

INITIAL AUDIT — weight_change_dataset.csv
Shape       : 100 rows × 13 columns
Duplicates  : 0

--- Columns ---
['Participant ID', 'Age', 'Gender', 'Current Weight (lbs)', 'BMR (Calories)', 'Daily Calories Consumed', 'Daily Caloric Surplus/Deficit', 'Weight Change (lbs)', 'Duration (weeks)', 'Physical Activity Level', 'Sleep Quality', 'Stress Level', 'Final Weight (lbs)']

--- Missing Values ---
Participant ID                   0
Age                              0
Gender                           0
Current Weight (lbs)             0
BMR (Calories)                   0
Daily Calories Consumed          0
Daily Caloric Surplus/Deficit    0
Weight Change (lbs)              0
Duration (weeks)                 0
Physical Activity Level          0
Sleep Quality                    0
Stress Level                     0
Final Weight (lbs)               0
dtype: int64

--------- Sample -----------


,Participant ID,Age,Gender,Current Weight (lbs),BMR (Calories),Daily Calories Consumed,Daily Caloric Surplus/Deficit,Weight Change (lbs),Duration (weeks),Physical Activity Level,Sleep Quality,Stress Level,Final Weight (lbs)
0,1,56,M,228.4,3102.3,3916.0,813.7,0.2,1,Sedentary,Excellent,6,228.6
1,2,46,F,165.4,2275.5,3823.0,1547.5,2.4,6,Very Active,Excellent,6,167.8
2,3,32,F,142.8,2119.4,2785.4,666.0,1.4,7,Sedentary,Good,3,144.2
3,4,25,F,145.5,2181.3,2587.3,406.0,0.8,8,Sedentary,Fair,2,146.3
4,5,38,M,155.5,2463.8,3312.8,849.0,2.0,10,Lightly Active,Good,1,157.5


## 3. Rename & Type Standardization

Gender di-encode ke `Male`/`Female` (full string) buat konsistensi dengan schema dan agar tidak ambigu buat AI Engineer dan backend.

In [24]:
COLUMN_RENAME_MAP = {
    'Age'                    : 'age',
    'Gender'                 : 'gender',
    'Current Weight (lbs)'   : 'weight_lbs',   # sementara, akan dikonversi
    'Physical Activity Level': 'activity_level'
}

df = df.rename(columns=COLUMN_RENAME_MAP)

# Standardize gender ke full string (konsisten dengan data lain)
GENDER_MAP = {'Male': 'Male', 'Female': 'Female', 'M': 'Male', 'F': 'Female'}
df['gender'] = df['gender'].map(GENDER_MAP)

print('Gender values setelah standarisasi:', df['gender'].unique())
print('Activity Level values:', df['activity_level'].unique())
print('Missing gender:', df['gender'].isnull().sum())

Gender values setelah standarisasi: ['Male' 'Female']
Activity Level values: ['Sedentary' 'Very Active' 'Lightly Active' 'Moderately Active']
Missing gender: 0


## 4. Unit Conversion — Weight (lbs → kg)

In [25]:
LBS_TO_KG = 0.453592
df['weight_kg'] = (df['weight_lbs'] * LBS_TO_KG).round(2)

# Drop kolom lbs — tidak diperlukan di output akhir
df = df.drop(columns=['weight_lbs'])

print('Weight range (kg):')
print(df['weight_kg'].describe().round(2))

Weight range (kg):
count    100.00
mean      77.81
std       13.76
min       45.36
25%       69.69
50%       78.09
75%       87.30
max      108.05
Name: weight_kg, dtype: float64


## 5. Synthetic Height Generation

**SYNTHETIC DATA** — dataset tidak ada `height_cm`.
Nilai di-generate menggunakan distribusi normal berdasarkan statistik antropometri Indonesia.

**Referensi :**
Parameter rata-rata (*mean*) disesuaikan dengan statistik antropometri riil masyarakat Indonesia berdasarkan artikel ilmiah: *“Pola Tinggi Badan Penduduk Indonesia: Perbandingan Antar Wilayah dan Kelompok Umur”*
- Rata-rata tinggi pria Indonesia: 163 cm
- Rata-rata tinggi wanita Indonesia: 152 cm

In [26]:
HEIGHT_PARAMS = {
    'Male'  : {'mean': 163, 'std': 7},
    'Female': {'mean': 152, 'std': 6}
}

def generate_height(gender: str) -> float:
    params = HEIGHT_PARAMS.get(gender, {'mean': 164, 'std': 7})
    return round(np.random.normal(params['mean'], params['std']), 1)

df['height_cm'] = df['gender'].apply(generate_height)

# Validasi: tidak boleh ada tinggi badan ekstrem (<130 atau >220 cm)
df['height_cm'] = df['height_cm'].clip(130, 220)

print('Height distribution (cm):')
print(df.groupby('gender')['height_cm'].describe().round(1))

Height distribution (cm):
        count   mean  std    min    25%    50%    75%    max
gender                                                      
Female   43.0  151.7  5.7  136.3  148.8  152.5  155.6  161.4
Male     57.0  162.0  6.1  149.3  158.8  161.4  165.4  176.0


## 6. BMR Calculation — Mifflin-St Jeor Equation

Formula yang dipakai adalah Mifflin-St Jeor (1990) — direkomendasikan oleh Academy of Nutrition and Dietetics sebagai formula BMR paling akurat untuk populasi umum:

```
BMR (Pria)    = 10×weight_kg + 6.25×height_cm − 5×age + 5
BMR (Wanita)  = 10×weight_kg + 6.25×height_cm − 5×age − 161
```

Unit output: **kkal/hari**

In [27]:
def calculate_bmr(row: pd.Series) -> float:
    """
    Mifflin-St Jeor BMR Equation.
    Input: weight_kg, height_cm, age, gender (Male/Female)
    Output: BMR in kcal/day
    """
    base = (10 * row['weight_kg']) + (6.25 * row['height_cm']) - (5 * row['age'])
    adjustment = 5 if row['gender'] == 'Male' else -161
    return round(base + adjustment, 2)

df['bmr'] = df.apply(calculate_bmr, axis=1)

# Sanity check: BMR tidak boleh < 500 atau > 4000 kcal/hari untuk manusia dewasa normal
invalid_bmr = df[(df['bmr'] < 500) | (df['bmr'] > 4000)]
print(f'BMR di luar range wajar (500–4000): {len(invalid_bmr)} baris')

print('\nBMR summary (kcal/day):')
print(df.groupby('gender')['bmr'].describe().round(1))

BMR di luar range wajar (500–4000): 0 baris

BMR summary (kcal/day):
        count    mean    std     min     25%     50%     75%     max
gender                                                              
Female   43.0  1329.5  148.0   971.4  1225.4  1345.3  1397.8  1663.8
Male     57.0  1640.8  148.9  1316.9  1521.2  1633.0  1755.2  2004.9


## 7. TDEE Calculation

**TDEE = BMR × Activity Multiplier**

| Activity Level | Multiplier | Deskripsi |
|---|---|---|
| Sedentary | 1.200 | Tidak/jarang olahraga |
| Lightly Active | 1.375 | Olahraga ringan 1–3 hari/minggu |
| Moderately Active | 1.550 | Olahraga sedang 3–5 hari/minggu |
| Very Active | 1.725 | Olahraga berat 6–7 hari/minggu |

In [28]:
ACTIVITY_MULTIPLIER = {
    'Sedentary'        : 1.200,
    'Lightly Active'   : 1.375,
    'Moderately Active': 1.550,
    'Very Active'      : 1.725
}

df['activity_multiplier'] = df['activity_level'].map(ACTIVITY_MULTIPLIER)

# Audit: cek apakah ada activity_level yang tidak terpetakan
unmapped = df['activity_multiplier'].isnull().sum()
print(f'Activity level tidak terpetakan: {unmapped}')
if unmapped > 0:
    print('Nilai tidak dikenal:', df[df['activity_multiplier'].isnull()]['activity_level'].unique())

df['tdee'] = (df['bmr'] * df['activity_multiplier']).round(2)

print('\nTDEE summary (kcal/day):')
print(df['tdee'].describe().round(1))

Activity level tidak terpetakan: 0

TDEE summary (kcal/day):
count     100.0
mean     2205.3
std       427.8
min      1335.6
25%      1895.3
50%      2160.9
75%      2514.7
max      3230.6
Name: tdee, dtype: float64


## 8. Goal Assignment & Target Calorie

**Goal adjustment** berdasarkan TDEE:
| Goal | Adjustment | Dasar |
|---|---|---|
| `cutting` | TDEE − 500 kcal | Defisit standar untuk penurunan ~0.45 kg/minggu |
| `maintain` | TDEE | Tidak ada adjustment |
| `bulking` | TDEE + 300 kcal | Surplus moderat untuk kenaikan massa otot |

**Catatan:** Minimum `target_calorie` dibatasi 1200 kcal/hari untuk wanita dan 1500 kcal/hari untuk pria — batas keamanan medis minimum.

**Update dari versi sebelumnya:** Deficit cutting diubah dari ±300 → ±500 kcal untuk mengikuti standar klinis yang lebih umum diterima.

In [29]:
GOAL_ADJUSTMENT = {
    'cutting' : -500,
    'maintain':    0,
    'bulking' : +300
}

MIN_CALORIE = {'Male': 1500, 'Female': 1200}

# Untuk demo schema, goal di-assign secara acak
# Pada sistem produksi, goal berasal dari input pengguna
np.random.seed(RANDOM_SEED)
df['goal'] = np.random.choice(['cutting', 'maintain', 'bulking'], size=len(df))

def calculate_target_calorie(row: pd.Series) -> float:
    """
    Target kalori = TDEE + goal_adjustment,
    dengan batas minimum keamanan per gender.
    """
    raw_target = row['tdee'] + GOAL_ADJUSTMENT[row['goal']]
    min_cal = MIN_CALORIE.get(row['gender'], 1200)
    return round(max(raw_target, min_cal), 2)

df['target_calorie'] = df.apply(calculate_target_calorie, axis=1)

print('Target calorie distribution per goal:')
print(df.groupby('goal')['target_calorie'].describe().round(1))

Target calorie distribution per goal:
          count    mean    std     min     25%     50%     75%     max
goal                                                                  
bulking    31.0  2541.1  458.3  1635.6  2190.2  2523.2  2824.0  3407.6
cutting    33.0  1600.8  356.0  1200.0  1319.2  1500.0  1877.7  2490.9
maintain   36.0  2316.1  384.7  1463.2  2034.3  2292.0  2637.0  3230.6


## 9. Macronutrient Target Calculation

**Strategi distribusi makro:**
- **Protein:** 2.0 g per kg berat badan (mendukung preservasi massa otot, terutama saat cutting)
- **Fat:** 0.8 g per kg berat badan (minimal untuk fungsi hormonal dan absorbsi vitamin larut lemak)
- **Carbohydrate:** Sisa kalori setelah protein dan fat dialokasikan

Formula kkal:
- Protein: 4 kcal/g
- Fat: 9 kcal/g
- Carbohydrate: 4 kcal/g

In [30]:
PROTEIN_RATIO = 0.20
FAT_RATIO = 0.25
CARB_RATIO = 0.55

# Protein
df['protein_target_g'] = (
    (df['target_calorie'] * PROTEIN_RATIO) / 4
).round(2)

# Fat
df['fat_target_g'] = (
    (df['target_calorie'] * FAT_RATIO) / 9
).round(2)

# Carbohydrate
df['carb_target_g'] = (
    (df['target_calorie'] * CARB_RATIO) / 4
).round(2)

df['carb_floored_flag'] = 0

print('Macro target summary (g/day):')
print(df[['protein_target_g', 'fat_target_g', 'carb_target_g']].describe().round(1))

Macro target summary (g/day):
       protein_target_g  fat_target_g  carb_target_g
count             100.0         100.0          100.0
mean              107.5          59.7          295.6
std                28.1          15.6           77.2
min                60.0          33.3          165.0
25%                85.8          47.6          235.9
50%               107.3          59.6          295.2
75%               127.1          70.6          349.6
max               170.4          94.6          468.5


## 10. Macro Validation — Cross Check

Validasi: total kalori dari makro harus mendekati `target_calorie`. Toleransi: ±100 kcal (karena ada rounding dan carb flooring).

In [31]:
df['total_macro_kcal'] = (
    df['protein_target_g'] * 4 +
    df['fat_target_g']     * 9 +
    df['carb_target_g']    * 4
).round(2)

df['macro_cal_diff'] = (df['total_macro_kcal'] - df['target_calorie']).abs().round(2)

MACRO_TOLERANCE = 100
out_of_tolerance = df[df['macro_cal_diff'] > MACRO_TOLERANCE]
print(f'Baris dengan selisih makro > {MACRO_TOLERANCE} kcal: {len(out_of_tolerance)}')
if len(out_of_tolerance) > 0:
    print(out_of_tolerance[['gender','goal','target_calorie','total_macro_kcal','macro_cal_diff']].head())

print('\n Macro validation selesai.')

Baris dengan selisih makro > 100 kcal: 0

 Macro validation selesai.


## 11. Allergy Vector Placeholder

Kolom `allergy_vector` adalah **wajib** per schema NutriMatch.

Pada dataset simulasi ini, `allergy_vector` di-set sebagai **placeholder string kosong** karena alergi adalah input pengguna yang tidak tersedia di dataset latih.

**Representasi produksi** (yang akan diimplementasikan saat integrasi dengan input pengguna):
```python
# Binary vector: [gluten, dairy, nuts, peanut, seafood, egg, soy, other]
# Contoh: pengguna alergi gluten dan seafood → '1,0,0,0,1,0,0,0'
```

In [32]:
# Placeholder: tidak ada alergi (semua 0)
# Format: 'gluten,dairy,nuts,peanut,seafood,egg,soy,other'
ALLERGY_CATEGORIES = ['gluten', 'dairy', 'nuts', 'peanut', 'seafood', 'egg', 'soy', 'other']
N_ALLERGY = len(ALLERGY_CATEGORIES)

df['allergy_vector'] = ','.join(['0'] * N_ALLERGY)

print(f'Allergy categories ({N_ALLERGY}): {ALLERGY_CATEGORIES}')
print(f'Placeholder allergy_vector: "{df["allergy_vector"].iloc[0]}"')

Allergy categories (8): ['gluten', 'dairy', 'nuts', 'peanut', 'seafood', 'egg', 'soy', 'other']
Placeholder allergy_vector: "0,0,0,0,0,0,0,0"


## 12. Final Column Selection & Export

In [33]:
# Kolom wajib schema
CORE_COLUMNS = [
    'age',
    'gender',
    'height_cm',
    'weight_kg',
    'activity_level',
    'goal',
    'bmr',
    'tdee',
    'target_calorie',
    'protein_target_g',
    'fat_target_g',
    'carb_target_g',
    'allergy_vector'
]

# Kolom audit
AUDIT_COLUMNS = [
    'carb_floored_flag',
    'total_macro_kcal',
    'macro_cal_diff'
]

user_profile_features_schema = df[CORE_COLUMNS + AUDIT_COLUMNS].copy()

print('=' * 55)
print('USER PROFILE FEATURES SCHEMA — SUMMARY')
print('=' * 55)
print(f'Total rows        : {len(user_profile_features_schema)}')
print(f'Total columns     : {len(user_profile_features_schema.columns)}')
print(f'Missing values    : {user_profile_features_schema.isnull().sum().sum()}')
print(f'Carb floored rows : {user_profile_features_schema["carb_floored_flag"].sum()}')
print()
print('--- Distribusi Statistik ---')
print(user_profile_features_schema[['bmr','tdee','target_calorie','protein_target_g','fat_target_g','carb_target_g']].describe().round(1))
print()
user_profile_features_schema.head()

USER PROFILE FEATURES SCHEMA — SUMMARY
Total rows        : 100
Total columns     : 16
Missing values    : 0
Carb floored rows : 0

--- Distribusi Statistik ---
          bmr    tdee  target_calorie  protein_target_g  fat_target_g  \
count   100.0   100.0           100.0             100.0         100.0   
mean   1507.0  2205.3          2149.8             107.5          59.7   
std     214.1   427.8           561.6              28.1          15.6   
min     971.4  1335.6          1200.0              60.0          33.3   
25%    1364.7  1895.3          1715.3              85.8          47.6   
50%    1506.4  2160.9          2146.8             107.3          59.6   
75%    1680.2  2514.7          2542.3             127.1          70.6   
max    2004.9  3230.6          3407.6             170.4          94.6   

       carb_target_g  
count          100.0  
mean           295.6  
std             77.2  
min            165.0  
25%            235.9  
50%            295.2  
75%            349.6 

,age,gender,height_cm,weight_kg,activity_level,goal,bmr,tdee,target_calorie,protein_target_g,fat_target_g,carb_target_g,allergy_vector,carb_floored_flag,total_macro_kcal,macro_cal_diff
0,56,Male,166.5,103.60,Sedentary,bulking,1801.62,2161.94,2461.94,123.10,68.39,338.52,"0,0,0,0,0,0,0,0",0,2461.99,0.05
1,46,Female,151.2,75.02,Very Active,cutting,1304.20,2249.75,1749.75,87.49,48.60,240.59,"0,0,0,0,0,0,0,0",0,1749.72,0.03
2,32,Female,155.9,64.77,Sedentary,bulking,1301.07,1561.28,1861.28,93.06,51.70,255.93,"0,0,0,0,0,0,0,0",0,1861.26,0.02
3,25,Female,161.1,66.00,Sedentary,bulking,1380.88,1657.06,1957.06,97.85,54.36,269.10,"0,0,0,0,0,0,0,0",0,1957.04,0.02
4,38,Male,161.4,70.53,Lightly Active,cutting,1529.05,2102.44,1602.44,80.12,44.51,220.34,"0,0,0,0,0,0,0,0",0,1602.43,0.01


In [35]:
OUTPUT_PATH = 'user_profile_features_schema.csv'
user_profile_features_schema.to_csv(OUTPUT_PATH, index=False)
print(f'Berhasil disimpan: {OUTPUT_PATH}')
print(f'   Shape: {user_profile_features_schema.shape}')

Berhasil disimpan: user_profile_features_schema.csv
   Shape: (100, 16)
